# W08 — Teacher (soluciones)

Para demos y revisión.

In [1]:
import os
os.chdir("..")
os.getcwd()

'/Users/itnas/Downloads/USB llaves/2704'

## Setup

In [2]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path(".").resolve()
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"

if not RAW_CSV.exists():
    raise FileNotFoundError(f"Missing {RAW_CSV}. Run W01/W02 download first.")

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'", "''") + "'"

con = duckdb.connect(str(DB_PATH))

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

con.sql("SELECT COUNT(*) AS n_raw FROM raw_ps").show()

┌───────┐
│ n_raw │
│ int64 │
├───────┤
│  6087 │
└───────┘



## Parte A — limpieza

In [3]:
con.sql("SELECT discoverymethod, COUNT(*) AS n FROM raw_ps WHERE discoverymethod IS NOT NULL GROUP BY discoverymethod ORDER BY n DESC LIMIT 15").show()
con.sql("SELECT COUNT(DISTINCT discoverymethod) AS n_unique_methods FROM raw_ps WHERE discoverymethod IS NOT NULL").show()

┌───────────────────────────────┬───────┐
│        discoverymethod        │   n   │
│            varchar            │ int64 │
├───────────────────────────────┼───────┤
│ Transit                       │  4488 │
│ Radial Velocity               │  1161 │
│ Microlensing                  │   265 │
│ Imaging                       │    91 │
│ Transit Timing Variations     │    39 │
│ Eclipse Timing Variations     │    17 │
│ Orbital Brightness Modulation │     9 │
│ Pulsar Timing                 │     8 │
│ Astrometry                    │     6 │
│ Pulsation Timing Variations   │     2 │
│ Disk Kinematics               │     1 │
└───────────────────────────────┴───────┘
  11 rows                     2 columns

┌──────────────────┐
│ n_unique_methods │
│      int64       │
├──────────────────┤
│               11 │
└──────────────────┘



In [4]:
con.execute("DROP TABLE IF EXISTS method_map")
con.execute("CREATE TABLE method_map(raw_method VARCHAR, canonical_method VARCHAR)")
con.execute("""INSERT INTO method_map VALUES
  ('Transit', 'transit'),
  ('transit', 'transit'),
  (' TRANSIT ', 'transit'),
  ('Radial Velocity', 'radial_velocity'),
  ('radial velocity', 'radial_velocity'),
  ('Imaging', 'imaging'),
  ('Microlensing', 'microlensing'),
  ('Timing', 'timing')""")
con.sql("SELECT * FROM method_map").show()

┌─────────────────┬──────────────────┐
│   raw_method    │ canonical_method │
│     varchar     │     varchar      │
├─────────────────┼──────────────────┤
│ Transit         │ transit          │
│ transit         │ transit          │
│  TRANSIT        │ transit          │
│ Radial Velocity │ radial_velocity  │
│ radial velocity │ radial_velocity  │
│ Imaging         │ imaging          │
│ Microlensing    │ microlensing     │
│ Timing          │ timing           │
└─────────────────┴──────────────────┘



In [5]:
con.execute("DROP TABLE IF EXISTS silver_planet_v2")
con.execute("""CREATE TABLE silver_planet_v2 AS
WITH base AS (
  SELECT
    pl_name, hostname, discoverymethod, disc_year, sy_snum, sy_pnum, sy_dist, ra, dec,
    pl_orbper, pl_rade, pl_bmasse, pl_eqt, st_teff, st_rad, st_mass,
    LOWER(TRIM(hostname)) AS hostname_clean,
    NULLIF(TRIM(discoverymethod), '') AS discoverymethod_norm
  FROM raw_ps
  WHERE pl_name IS NOT NULL
    AND hostname IS NOT NULL
    AND (disc_year IS NULL OR (disc_year BETWEEN 1980 AND 2026))
    AND (pl_rade  IS NULL OR (pl_rade  > 0 AND pl_rade  <= 30))
    AND (pl_bmasse IS NULL OR (pl_bmasse > 0))
),
mapped AS (
  SELECT
    b.*,
    COALESCE(m.canonical_method, LOWER(TRIM(b.discoverymethod_norm))) AS discoverymethod_clean,
    CASE
      WHEN b.disc_year IS NULL THEN NULL
      WHEN b.disc_year < 1990 THEN 'pre_1990'
      WHEN b.disc_year < 2000 THEN '1990s'
      WHEN b.disc_year < 2010 THEN '2000s'
      WHEN b.disc_year < 2020 THEN '2010s'
      ELSE '2020s'
    END AS disc_era
  FROM base b
  LEFT JOIN method_map m
    ON m.raw_method = b.discoverymethod
)
SELECT
  pl_name, hostname, discoverymethod, disc_year, sy_snum, sy_pnum, sy_dist, ra, dec,
  pl_orbper, pl_rade, pl_bmasse, pl_eqt, st_teff, st_rad, st_mass,
  hostname_clean, discoverymethod_clean, disc_era
FROM mapped""")
con.sql("SELECT COUNT(*) AS n_rows FROM silver_planet_v2").show()
con.sql("SELECT COUNT(*) AS n_null_hosts FROM silver_planet_v2 WHERE hostname_clean IS NULL").show()
con.sql("SELECT discoverymethod_clean, COUNT(*) AS n FROM silver_planet_v2 WHERE discoverymethod_clean IS NOT NULL GROUP BY discoverymethod_clean ORDER BY n DESC LIMIT 15").show()

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│   6081 │
└────────┘

┌──────────────┐
│ n_null_hosts │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘

┌───────────────────────────────┬───────┐
│     discoverymethod_clean     │   n   │
│            varchar            │ int64 │
├───────────────────────────────┼───────┤
│ transit                       │  4487 │
│ radial_velocity               │  1161 │
│ microlensing                  │   265 │
│ imaging                       │    86 │
│ transit timing variations     │    39 │
│ eclipse timing variations     │    17 │
│ orbital brightness modulation │     9 │
│ pulsar timing                 │     8 │
│ astrometry                    │     6 │
│ pulsation timing variations   │     2 │
│ disk kinematics               │     1 │
└───────────────────────────────┴───────┘
  11 rows                     2 columns



## Parte B — Many-to-Many

In [6]:
# Idempotencia (FK-safe): dropea puente primero.
con.execute("DROP TABLE IF EXISTS planet_method_demo")
con.execute("DROP TABLE IF EXISTS method_demo")
con.execute("DROP TABLE IF EXISTS planet_demo")

con.execute("""
CREATE TABLE planet_demo(
  planet_id INTEGER PRIMARY KEY,
  name VARCHAR NOT NULL
);
""")

con.execute("""
CREATE TABLE method_demo(
  method_id INTEGER PRIMARY KEY,
  method_name VARCHAR NOT NULL UNIQUE
);
""")

con.execute("""
CREATE TABLE planet_method_demo(
  planet_id INTEGER NOT NULL,
  method_id INTEGER NOT NULL,
  PRIMARY KEY (planet_id, method_id),
  FOREIGN KEY (planet_id) REFERENCES planet_demo(planet_id),
  FOREIGN KEY (method_id) REFERENCES method_demo(method_id)
);
""")

con.execute("""
INSERT INTO planet_demo VALUES
  (1, 'Aurelia'),
  (2, 'Borealis'),
  (3, 'Cetus'),
  (4, 'Draco');
""")

con.execute("""
INSERT INTO method_demo VALUES
  (10, 'transit'),
  (20, 'radial_velocity'),
  (30, 'imaging');
""")

con.execute("""
INSERT INTO planet_method_demo VALUES
  (1, 10),
  (1, 20),
  (2, 10),
  (3, 30),
  (4, 20),
  (4, 30);
""")

# (Demo) La PK compuesta evita duplicados de pares (planet_id, method_id)
try:
    con.execute("INSERT INTO planet_method_demo VALUES (1, 10)")
except Exception as e:
    print("Esperado (PK compuesta evita duplicado):", str(e).splitlines()[0])

# Q1: # planetas por método (DISTINCT evita doble conteo)
q1 = """
SELECT
  m.method_name,
  COUNT(DISTINCT pm.planet_id) AS n_planets
FROM method_demo m
JOIN planet_method_demo pm
  ON pm.method_id = m.method_id
GROUP BY m.method_name
ORDER BY n_planets DESC, m.method_name;
"""
con.sql(q1).show()

# Q2: # métodos por planeta
q2 = """
SELECT
  p.name,
  COUNT(DISTINCT pm.method_id) AS n_methods
FROM planet_demo p
JOIN planet_method_demo pm
  ON pm.planet_id = p.planet_id
GROUP BY p.name
ORDER BY n_methods DESC, p.name;
"""
con.sql(q2).show()

# Check de cardinalidad: pares duplicados en la link table (debe ser 0)
con.sql("""
SELECT COUNT(*) AS n_duplicate_pairs
FROM (
  SELECT planet_id, method_id, COUNT(*) AS c
  FROM planet_method_demo
  GROUP BY planet_id, method_id
  HAVING COUNT(*) > 1
);
""").show()

Esperado (PK compuesta evita duplicado): Constraint Error: Duplicate key "planet_id: 1, method_id: 10" violates primary key constraint.
┌─────────────────┬───────────┐
│   method_name   │ n_planets │
│     varchar     │   int64   │
├─────────────────┼───────────┤
│ imaging         │         2 │
│ radial_velocity │         2 │
│ transit         │         2 │
└─────────────────┴───────────┘

┌──────────┬───────────┐
│   name   │ n_methods │
│ varchar  │   int64   │
├──────────┼───────────┤
│ Aurelia  │         2 │
│ Draco    │         2 │
│ Borealis │         1 │
│ Cetus    │         1 │
└──────────┴───────────┘

┌───────────────────┐
│ n_duplicate_pairs │
│       int64       │
├───────────────────┤
│                 0 │
└───────────────────┘



## #métodos por planeta (quién tiene más evidencia/métodos)

In [8]:
con.sql("""
SELECT
  p.name,
  COUNT(DISTINCT m.method_id) AS n_methods
FROM planet_demo p
JOIN planet_method_demo pm
  ON pm.planet_id = p.planet_id
JOIN method_demo m
  ON m.method_id = pm.method_id
GROUP BY p.name
ORDER BY n_methods DESC, p.name;
""").show()

┌──────────┬───────────┐
│   name   │ n_methods │
│ varchar  │   int64   │
├──────────┼───────────┤
│ Aurelia  │         2 │
│ Draco    │         2 │
│ Borealis │         1 │
│ Cetus    │         1 │
└──────────┴───────────┘



## Co-ocurrencia: pares de métodos que aparecen juntos en el mismo planet

In [10]:
con.sql("""
SELECT
  m1.method_name AS method_a,
  m2.method_name AS method_b,
  COUNT(DISTINCT pm1.planet_id) AS n_planets_with_both
FROM planet_method_demo pm1
JOIN planet_method_demo pm2
  ON pm1.planet_id = pm2.planet_id
 AND pm1.method_id < pm2.method_id       -- evita duplicados (A,B) vs (B,A)
JOIN method_demo m1
  ON m1.method_id = pm1.method_id
JOIN method_demo m2
  ON m2.method_id = pm2.method_id
JOIN planet_demo p
  ON p.planet_id = pm1.planet_id         -- (une 3 tablas, aunque p no salga en SELECT)
GROUP BY method_a, method_b
ORDER BY n_planets_with_both DESC, method_a, method_b;
""").show()

┌─────────────────┬─────────────────┬─────────────────────┐
│    method_a     │    method_b     │ n_planets_with_both │
│     varchar     │     varchar     │        int64        │
├─────────────────┼─────────────────┼─────────────────────┤
│ radial_velocity │ imaging         │                   1 │
│ transit         │ radial_velocity │                   1 │
└─────────────────┴─────────────────┴─────────────────────┘



## Top planeta ‘más diverso’: planeta con más métodos (y muestra cuáles son)

In [14]:
con.sql("""
WITH counts AS (
  SELECT
    p.planet_id,
    p.name,
    COUNT(DISTINCT pm.method_id) AS n_methods
  FROM planet_demo p
  JOIN planet_method_demo pm
    ON pm.planet_id = p.planet_id
  JOIN method_demo m
    ON m.method_id = pm.method_id
  GROUP BY p.planet_id, p.name
),
top1 AS (
  SELECT *
  FROM counts
  ORDER BY n_methods DESC, name
  LIMIT 1
)
SELECT
  t.name,
  t.n_methods,
  m.method_name
FROM top1 t
JOIN planet_method_demo pm
  ON pm.planet_id = t.planet_id
JOIN method_demo m
  ON m.method_id = pm.method_id
ORDER BY m.method_name;
""").show()

┌─────────┬───────────┬─────────────────┐
│  name   │ n_methods │   method_name   │
│ varchar │   int64   │     varchar     │
├─────────┼───────────┼─────────────────┤
│ Aurelia │         2 │ radial_velocity │
│ Aurelia │         2 │ transit         │
└─────────┴───────────┴─────────────────┘



## Cobertura: porcentaje de planetas cubiertos por cada método

In [15]:
con.sql("""
WITH total AS (
  SELECT COUNT(*) AS n_planets_total
  FROM planet_demo
),
by_method AS (
  SELECT
    m.method_name,
    COUNT(DISTINCT p.planet_id) AS n_planets
  FROM method_demo m
  JOIN planet_method_demo pm
    ON pm.method_id = m.method_id
  JOIN planet_demo p
    ON p.planet_id = pm.planet_id
  GROUP BY m.method_name
)
SELECT
  b.method_name,
  b.n_planets,
  ROUND(100.0 * b.n_planets / t.n_planets_total, 2) AS pct_coverage
FROM by_method b
CROSS JOIN total t
ORDER BY pct_coverage DESC, b.method_name;
""").show()

┌─────────────────┬───────────┬──────────────┐
│   method_name   │ n_planets │ pct_coverage │
│     varchar     │   int64   │    double    │
├─────────────────┼───────────┼──────────────┤
│ imaging         │         2 │         50.0 │
│ radial_velocity │         2 │         50.0 │
│ transit         │         2 │         50.0 │
└─────────────────┴───────────┴──────────────┘



## Quality Check Real Planetas sin métodos (detectar huérfanos en la práctica)

In [16]:
con.sql("""
SELECT
  p.planet_id,
  p.name
FROM planet_demo p
LEFT JOIN planet_method_demo pm
  ON pm.planet_id = p.planet_id
LEFT JOIN method_demo m
  ON m.method_id = pm.method_id
WHERE pm.planet_id IS NULL
ORDER BY p.planet_id;
""").show()

┌───────────┬─────────┐
│ planet_id │  name   │
│   int32   │ varchar │
└───────────┴─────────┘
        0 rows       



## Métodos sin planetas (otro check de calidad)

In [17]:
con.sql("""
SELECT
  m.method_id,
  m.method_name
FROM method_demo m
LEFT JOIN planet_method_demo pm
  ON pm.method_id = m.method_id
LEFT JOIN planet_demo p
  ON p.planet_id = pm.planet_id
WHERE pm.method_id IS NULL
ORDER BY m.method_id;
""").show()

┌───────────┬─────────────┐
│ method_id │ method_name │
│   int32   │   varchar   │
└───────────┴─────────────┘
          0 rows         



## Relaciones totales y densidad del grafo (potente para hablar de topología)

In [20]:
con.sql("""
WITH counts AS (
  SELECT
    (SELECT COUNT(*) FROM planet_demo) AS n_planets,
    (SELECT COUNT(*) FROM method_demo) AS n_methods,
    (SELECT COUNT(*) FROM planet_method_demo) AS n_links
)
SELECT
  n_planets,
  n_methods,
  n_links,
  ROUND(1.0 * n_links / (n_planets * n_methods), 4) AS density
FROM counts;
""").show()
# Densidad = edges / (|P|*|M|)

┌───────────┬───────────┬─────────┬─────────┐
│ n_planets │ n_methods │ n_links │ density │
│   int64   │   int64   │  int64  │ double  │
├───────────┼───────────┼─────────┼─────────┤
│         4 │         3 │       6 │     0.5 │
└───────────┴───────────┴─────────┴─────────┘

